In [1]:
!pip install -q -U bitsandbytes transformers peft accelerate trl datasets

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 1. Force everything to float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Math in fp16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16, # Weights in fp16
)

# 2. Hard-reset the model config to avoid any BF16 leftovers
model.config.torch_dtype = torch.float16
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 3. PEFT Setup
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

dataset = load_dataset('json', data_files='train.jsonl', split='train')

sft_config = SFTConfig(
    output_dir="./tinyllama-finance",
    dataset_text_field="instruction",
    max_length=512,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    fp16=False,
    bf16=False,
    save_strategy="epoch",
    gradient_checkpointing=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    #  peft_config is omitted because we already applied it in the cell above
)

trainer.train()
model.save_pretrained("./adapters")
print(" SUCCESS! The T4 is finally training.")

Step,Training Loss
5,0.148849
10,0.124272
15,0.117899
20,0.115149
25,0.118721
30,0.128580
35,0.119848
40,0.121245
45,0.113221
50,0.118883


 SUCCESS! The T4 is finally training.


In [5]:
# Zip the folders so they can be downloaded as one file
!zip -r adapters.zip /content/adapters
!zip -r tinyllama-finance.zip /content/tinyllama-finance

  adding: content/adapters/ (stored 0%)
  adding: content/adapters/README.md (deflated 65%)
  adding: content/adapters/adapter_config.json (deflated 57%)
  adding: content/adapters/adapter_model.safetensors (deflated 22%)
  adding: content/tinyllama-finance/ (stored 0%)
  adding: content/tinyllama-finance/checkpoint-65/ (stored 0%)
  adding: content/tinyllama-finance/checkpoint-65/trainer_state.json (deflated 73%)
  adding: content/tinyllama-finance/checkpoint-65/scheduler.pt (deflated 61%)
  adding: content/tinyllama-finance/checkpoint-65/tokenizer_config.json (deflated 44%)
  adding: content/tinyllama-finance/checkpoint-65/training_args.bin (deflated 53%)
  adding: content/tinyllama-finance/checkpoint-65/README.md (deflated 65%)
  adding: content/tinyllama-finance/checkpoint-65/tokenizer.json (deflated 85%)
  adding: content/tinyllama-finance/checkpoint-65/optimizer.pt (deflated 20%)
  adding: content/tinyllama-finance/checkpoint-65/rng_state.pth (deflated 26%)
  adding: content/tiny

In [6]:
from google.colab import files

files.download('adapters.zip')
files.download('tinyllama-finance.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>